# Guitar TART Experiment — Colab Training

Beats TART 76% technique accuracy using MERT finetuning on Guitar-TECHS.

**Before running:** Get the Guitar-TECHS download URL from https://guitar-techs.github.io/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/guitar-tart', exist_ok=True)
%cd /content/drive/MyDrive/guitar-tart

In [ ]:
import os
if not os.path.exists('guitar-tart-experiments'):
    !git clone https://github.com/euisuh/guitar-tart-experiments.git
%cd guitar-tart-experiments
!pip install -r requirements.txt -q

In [ ]:
# Set your Guitar-TECHS download URL here
GUITAR_TECHS_URL = ''  # Get from https://guitar-techs.github.io/
assert GUITAR_TECHS_URL, 'Set GUITAR_TECHS_URL above'
!bash scripts/setup_dataset.sh {GUITAR_TECHS_URL}

In [ ]:
import wandb
wandb.login()

In [ ]:
# Train MERT-330M on CUDA
!python training/train.py \
    --model-name m-a-p/MERT-v1-330M \
    --batch-size 32 \
    --epochs 30 \
    --precision fp16 \
    --checkpoint-dir checkpoints/mert-330m \
    --run-name mert-330m-run1

In [ ]:
# Train MLP baseline for comparison
!python training/train.py \
    --model-type mlp \
    --batch-size 32 \
    --epochs 30 \
    --checkpoint-dir checkpoints/mlp-baseline \
    --run-name mlp-baseline-run1

In [ ]:
# Evaluate MERT
!python eval/evaluate.py \
    --checkpoint checkpoints/mert-330m/best.pt \
    --confusion-matrix-out mert_confusion.png

from IPython.display import Image
Image('mert_confusion.png')

In [ ]:
# Evaluate MLP baseline
!python eval/evaluate.py \
    --checkpoint checkpoints/mlp-baseline/best.pt \
    --confusion-matrix-out mlp_confusion.png

Image('mlp_confusion.png')